# ComfyUI on a free cloud GPU (Kaggle) + public tunnel

Use this when your PC has no strong NVIDIA GPU.
1. kaggle.com → New Notebook → *File > Import Notebook* → this file.
2. Right panel: **Accelerator = GPU T4 x2**, **Internet = On** (requires phone verification, free; ~30 GPU hours/week).
3. Run all cells. The last cell prints a `https://xxxx.trycloudflare.com` URL.
4. On your Windows PC: put that URL in `%USERPROFILE%\.comfy_vfx\config.json` → `"comfy_url"` (or tell Claude Code to do it).

The URL changes every session. Anyone with the URL can use your ComfyUI while it runs — don't share it.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -q -r requirements.txt

In [ ]:
# Models (add the ones your workflows use)
!wget -q -nc -O models/checkpoints/v1-5-pruned-emaonly.safetensors https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly.safetensors
!wget -q -nc -O models/upscale_models/RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
!ls -lh models/checkpoints models/upscale_models

In [ ]:
!wget -q -nc -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared

In [ ]:
import subprocess, time, re, threading
comfy = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                         stdout=open("/kaggle/working/comfy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(20)
tun = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n=== comfy_url ===\n" + m.group(0) + "\n")
        break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
# Keep this cell running; stop the notebook when done to save GPU quota.
comfy.wait()